# ComfyUI HTTP Client Test (`image_z_image_turbo`)

This notebook submits a ComfyUI workflow over HTTP and retrieves generated images.

It uses standard ComfyUI endpoints directly:
- `POST /prompt`
- `GET /history/{prompt_id}`
- `GET /view`

Prerequisites:
- ComfyUI server must be running and accessible.
- Enable the `Dev Mode` in the ComfyUI server to allow API access.
- Export the JSON file for the workflow from the ComfyUI UI, for example `image_z_image_turbo` and place it in the same directory as this notebook.
This could be done by going to the ComfyUI UI, clicking on the `image_z_image_turbo` workflow, and then clicking the `Export (API)` button to save the workflow JSON file as shown in the following screenshot.

![Export Workflow JSON](./images/export_workflow_template.png)

First of all, let's get the FQDN of the ComfyUI server.

In [1]:
aca_comfyui_cu126_a100_fqdn = ! terraform output -raw aca_comfyui_cu126_a100_fqdn
aca_comfyui_cu126_a100_fqdn = aca_comfyui_cu126_a100_fqdn.n
print("ComfyUI Endpoint:", aca_comfyui_cu126_a100_fqdn)

ComfyUI Endpoint: comfyui-cu126-a100.gentlebeach-178a8d2a.swedencentral.azurecontainerapps.io


In [ ]:
import copy
import json
import time
from pathlib import Path
from urllib import parse, request
from IPython.display import Image, display

# Configuration
COMFYUI_BASE_URL = f"http://{aca_comfyui_cu126_a100_fqdn}"
WORKFLOW_FILE = "image_z_image_turbo.json"

# Overrides (set to None to skip)
PROMPT = "BMW X5 in dark red, parked on street at night"
SEED = None
STEPS = 10
WIDTH = 1920
HEIGHT = 1080

class Client:
    def __init__(self, url, timeout=120):
        self.url = url.rstrip("/")
        self.timeout = timeout
    
    def post(self, path, body):
        req = request.Request(
            f"{self.url}{path}",
            data=json.dumps(body).encode(),
            headers={"Content-Type": "application/json"},
            method="POST"
        )
        with request.urlopen(req, timeout=self.timeout) as r:
            return json.loads(r.read())
    
    def get(self, path, params=None):
        url = f"{self.url}{path}"
        if params:
            url += "?" + parse.urlencode(params)
        with request.urlopen(url, timeout=self.timeout) as r:
            return r.read()
    
    def queue(self, workflow):
        resp = self.post("/prompt", {"prompt": workflow})
        return resp["prompt_id"]
    
    def history(self, pid):
        return self.get(f"/history/{pid}")
    
    def image(self, fname, subfolder=""):
        return self.get("/view", {"filename": fname, "subfolder": subfolder, "type": "output"})

print("Client ready")

Client ready


In [ ]:
# Load and apply overrides
workflow = json.loads(Path(WORKFLOW_FILE).read_text())

for nid, node in workflow.items():
    ct = node.get("class_type")
    if ct == "CLIPTextEncode" and PROMPT:
        node["inputs"]["text"] = PROMPT
    if ct == "KSampler":
        if SEED is not None:
            node["inputs"]["seed"] = SEED
        if STEPS is not None:
            node["inputs"]["steps"] = STEPS
    if ct == "EmptySD3LatentImage":
        if WIDTH is not None:
            node["inputs"]["width"] = WIDTH
        if HEIGHT is not None:
            node["inputs"]["height"] = HEIGHT

# Queue, wait for images, display
client = Client(COMFYUI_BASE_URL)
pid = client.queue(workflow)
print(f"Queued: {pid}")

# Wait for result
for _ in range(180):
    hist = json.loads(client.history(pid))
    if pid in hist and hist[pid].get("outputs"):
        break
    time.sleep(1)

# Get first image
outputs = hist[pid]["outputs"]
img_meta = outputs[list(outputs.keys())[0]]["images"][0]
fname = img_meta["filename"]
subfolder = img_meta.get("subfolder", "")

img_bytes = client.image(fname, subfolder)
display(Image(data=img_bytes))

# Save locally
Path("generated").mkdir(exist_ok=True)
Path(f"generated/{fname}").write_bytes(img_bytes)
print(f"Saved: generated/{fname}")

Queued: 018e8791-ae95-436b-804c-ca2c8b0e7314
